# nb43 - Aggregate-fraction supervision and quantile calibration (H7 + H8)

**Error analysis.** nb28 fig9: sigma_eff is 0.0281 in the low pileup-contamination tertile vs 0.1082 in the high one; nb37: contamination is the worst error axis at E>17 GeV (0.041). The gate in the subtract-then-calibrate readout is identified only through the calibrated sum - no direct signal tells it WHICH energy to keep.

**Question.** Can supervision we already possess - the exact per-event pileup fraction and the per-event residual width - sharpen the gate without per-cell labels?

**Hypotheses.** H7: adding a loss that ties the gated energy fraction to the known per-event signal fraction improves per-bin sigma_eff (learning-from-label-proportions; our fractions are exact per event, stronger than LLP assumes). H8: quantile heads plus width-binned recalibration improve the effective width (CMS b-jet regression reports 12-15%).

**Research.** LLP: Dery et al., JHEP 2017, arXiv:1702.00414; sPlot-weighted ML: Borisyak & Kazeev, JINST 2019, arXiv:1905.11719; identifiability of learning from aggregate observations: Zhang et al., NeurIPS 2020, arXiv:2004.06316. Quantile route: CMS, CSBS 4 (2020) 10, arXiv:1912.06046; Gaussian Ansatz: Gambhir, Nachman, Thaler, PRL 129, 082001.

**Proof criterion.** 2 seeds per config, pure-minbias training identical to nb32; a config wins if its 2-seed mean beats `base` by >0.002 overall or in any E>17 GeV bin on the fixed test split; smaller differences are noise; ties resolved on val only. The fraction target uses a median containment prior from clean data, so it is exact only on average - if `frac` is flat, prior noise is the first confound to check.

**Anchors.** nb32 W4 singles 0.0465 +/- 0.0006, 3-seed ens 0.0450; per-bin targets 0.06/0.045/0.035/0.032/0.030/0.030.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLEANF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB43_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB43_MODE', 'full')
if MODE == 'smoke': MB, CLEANF = MB[:8], CLEANF[:4]
THRESH = 2.49
print('device', DEVICE, '| mode', MODE, '|', len(MB), 'minbias files | THRESH', THRESH, 'MeV')

device cuda | mode full | 94 minbias files | THRESH 2.49 MeV


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5 or not ok[seed]: continue
            tf = cc['cell_times_front']; tb = cc['cell_times_back']
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[ok].astype(np.int16), dj=dj[ok].astype(np.int16),
                           e=e[ok].astype(np.float32),
                           fr=cc['cell_energies_front'][ok].astype(np.float32),
                           bk=cc['cell_energies_back'][ok].astype(np.float32),
                           tf=tf[ok].astype(np.float32), tb=tb[ok].astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
ME = build_grid(MB, 'minbias')
CE = build_grid(CLEANF, 'clean')
print(f'build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events
build 132s


In [3]:
def make_windows(W, EVS):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
        tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'], ev['reg'])); keep.append(i)
    return rows, np.array(keep)
def splits_for(keep):
    remap = -np.ones(len(ME), int); remap[keep] = np.arange(len(keep))
    a, b, t = split(len(ME))
    return (remap[a][remap[a] >= 0], remap[b][remap[b] >= 0], remap[t][remap[t] >= 0])

Containment prior from clean events: median in-window deposit fraction per (region, energy bin) converts Etrue into the expected in-window signal energy, giving the per-event fraction target f_true.

In [4]:
def containment_prior(crows):
    reg = np.array([r[4] for r in crows]); et = np.array([r[3] for r in crows])
    se = np.array([r[1] for r in crows]); ratio = se / (1000.0 * et)
    edges = np.quantile(et, np.linspace(0, 1, 7))
    table = np.full((len(PITCH), 6), np.nan)
    for g in range(len(PITCH)):
        for b in range(6):
            hi = edges[b+1] + (1e-9 if b == 5 else 0)
            m = (reg == g) & (et >= edges[b]) & (et < hi)
            if m.sum() >= 20: table[g, b] = np.median(ratio[m])
    fill = np.nanmedian(table)
    table = np.where(np.isfinite(table), table, fill)
    return table, edges

In [5]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
NG = 5; NC = 9
class SubNetFQ(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        gated = (w * ecell).sum(1, keepdim=True)
        base = self.la * torch.log1p(gated) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        q = base + self.head(torch.cat([p, g], 1))
        fbar = gated.squeeze(1) / (ecell.sum(1) + 1e-6)
        return q, fbar
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
def prep(W):
    rows, keep = make_windows(W, ME)
    ktr, kva, kte = splits_for(keep)
    crows, _ = make_windows(W, CE)
    ctable, cedges = containment_prior(crows)
    N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
    y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
    Et = np.array([r[3] for r in rows], np.float32)
    sumE = np.array([r[1] for r in rows], np.float32)
    reg = np.array([r[4] for r in rows])
    ebin = np.clip(np.searchsorted(cedges, Et, side='right') - 1, 0, 5)
    Ftrue = np.clip(ctable[reg, ebin] * 1000.0 * Et / np.maximum(sumE, EPS), 0.0, 1.0).astype(np.float32)
    X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
    G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
    for i, (tok, se, sde, et, rg) in enumerate(rows):
        n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
        e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
    la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
    G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
    cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
    mean = cont.mean(0); std = cont.std(0) + EPS
    X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
    T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
             G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
             E=torch.from_numpy(Eraw).to(DEVICE), F=torch.from_numpy(Ftrue).to(DEVICE))
    print(f'W={W}: N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, IN_DIM {IN_DIM}')
    print(f'f_true: median {np.median(Ftrue):.3f}, p10 {np.quantile(Ftrue, 0.1):.3f}, p90 {np.quantile(Ftrue, 0.9):.3f}')
    return T, y, Et, Ftrue, ktr, kva, kte, IN_DIM, float(la0), float(lb0)

In [6]:
LAM_FRAC = 0.25
def train_eval(T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0, config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(IN_DIM, la0, lb0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    QS = torch.tensor([0.25, 0.5, 0.75], device=DEVICE)
    ck = CKPT / f'nb43_{config}_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def pinball(q, yb):
        d = yb - q
        return torch.maximum(QS * d, (QS - 1) * d).mean()
    def loss_fn(q, fbar, yb, fb):
        if config in ('quant', 'fracquant'): L = pinball(q, yb)
        else: L = nn.functional.huber_loss(q[:, 1:2], yb, delta=CFG['huber_delta'])
        if config in ('frac', 'fracquant'): L = L + LAM_FRAC * nn.functional.mse_loss(fbar, fb)
        return L
    def run(idx):
        model.eval(); qs = []; fs = []
        with torch.no_grad():
            for b in batches(idx, 256, False):
                qq, ff = fwd(b); qs.append(qq.cpu().numpy()); fs.append(ff.cpu().numpy())
        return np.concatenate(qs), np.concatenate(fs)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                q, fbar = fwd(b)
                s += loss_fn(q, fbar, T['Y'][b], T['F'][b]).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad()
            q, fbar = fwd(b)
            loss_fn(q, fbar, T['Y'][b], T['F'][b]).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    qv, fv = run(kva); qt, ft = run(kte)
    if config in ('quant', 'fracquant'):
        wv = qv[:, 2] - qv[:, 0]; wt = qt[:, 2] - qt[:, 0]
        cuts = np.quantile(wv, [1/3, 2/3])
        gv = np.digitize(wv, cuts); gt = np.digitize(wt, cuts)
        pe = np.empty(len(kte))
        for g in range(3):
            if (gv == g).sum() < 10 or (gt == g).sum() == 0:
                a, b2 = np.polyfit(qv[:, 1], y[kva], 1)
            else:
                a, b2 = np.polyfit(qv[gv == g, 1], y[np.asarray(kva)[gv == g]], 1)
            pe[gt == g] = np.exp(a * qt[gt == g, 1] + b2)
    else:
        a, b2 = np.polyfit(qv[:, 1], y[kva], 1)
        pe = np.exp(a * qt[:, 1] + b2)
    return float(resolution(pe, Et[kte])['sigma_eff']), pe, ft

In [7]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
CONFIGS = ['base', 'frac', 'quant', 'fracquant']
JOBS = {'smoke': [(cfg, 0) for cfg in CONFIGS],
        'full': [(cfg, s) for cfg in CONFIGS for s in (0, 1)]}[MODE]
W = 4
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb43_fraction_quantile{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
T, y, Et, Ftrue, ktr, kva, kte, IN_DIM, la0, lb0 = prep(W)
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    t0 = time.time()
    sig, pe, ft = train_eval(T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0, config, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb43_pred{TAG}_{config}_s{seed}.npy', pe)
    np.save(OUT / f'nb43_fbar{TAG}_{config}_s{seed}.npy', ft)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t0))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

W=4: N 72554, tr/va/te 50787/10883/10884, IN_DIM 16
f_true: median 0.624, p10 0.301, p90 0.882


base seed 0: sigma_eff 0.0468 (432s)


base seed 1: sigma_eff 0.0464 (720s)


frac seed 0: sigma_eff 0.0505 (815s)


frac seed 1: sigma_eff 0.0476 (753s)


quant seed 0: sigma_eff 0.0446 (897s)


quant seed 1: sigma_eff 0.0445 (791s)


fracquant seed 0: sigma_eff 0.0457 (909s)


fracquant seed 1: sigma_eff 0.0445 (913s)


   config  seed  sigma_eff  elapsed
     base     0     0.0468      432
     base     1     0.0464      720
     frac     0     0.0505      815
     frac     1     0.0476      753
    quant     0     0.0446      897
    quant     1     0.0445      791
fracquant     0     0.0457      909
fracquant     1     0.0445      913


## Verdict: H7 (frac) and H8 (quant) vs the base control

Win = 2-seed mean beats base by >0.002 overall or in any E>17 GeV bin; smaller = noise. Gate diagnostic: correlation of the gated fraction with f_true tells whether the H7 loss actually steers the gate.

In [8]:
te_e = Et[kte]
print('anchors: nb32 W4 singles 0.0465 +/- 0.0006, 3-seed ens 0.0450 | targets 0.06/0.045/0.035/0.032/0.030/0.030')
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
stats = {}
for cfg in CONFIGS:
    preds = [np.load(OUT / f'nb43_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb43_pred{TAG}_{cfg}_s{s}.npy').exists()]
    fbars = [np.load(OUT / f'nb43_fbar{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb43_fbar{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    corr = np.mean([np.corrcoef(f, Ftrue[kte])[0, 1] for f in fbars])
    stats[cfg] = dict(mean=np.mean(sig), spread=np.std(sig), ens=resolution(ens, te_e)['sigma_eff'],
                      bins=perbin(ens), corr=corr, n=len(preds))
base = stats.get('base')
for cfg, st in stats.items():
    d = st['mean'] - base['mean'] if base else float('nan')
    flag = 'WIN' if d < -0.002 else ('HURT' if d > 0.002 else 'noise')
    print(f"{cfg:10s} mean {st['mean']:.4f} +/- {st['spread']:.4f} (n={st['n']}) | ens {st['ens']:.4f} | "
          f"d(base) {d:+.4f} [{flag}] | corr(fbar, f_true) {st['corr']:.3f}")
    print('           per-bin ' + ' / '.join(f'{b:.4f}' for b in st['bins']))
if base:
    for cfg in ('frac', 'quant', 'fracquant'):
        if cfg not in stats: continue
        db = [stats[cfg]['bins'][i] - base['bins'][i] for i in range(6)]
        hi = [i for i in range(2, 6) if db[i] < -0.002]
        print(f'{cfg}: E>17 bins beating base by >0.002: {hi if hi else "none"}')

anchors: nb32 W4 singles 0.0465 +/- 0.0006, 3-seed ens 0.0450 | targets 0.06/0.045/0.035/0.032/0.030/0.030


base       mean 0.0466 +/- 0.0002 (n=2) | ens 0.0451 | d(base) +0.0000 [noise] | corr(fbar, f_true) 0.917
           per-bin 0.0676 / 0.0497 / 0.0383 / 0.0387 / 0.0386 / 0.0403
frac       mean 0.0491 +/- 0.0014 (n=2) | ens 0.0474 | d(base) +0.0025 [HURT] | corr(fbar, f_true) 0.971
           per-bin 0.0703 / 0.0522 / 0.0413 / 0.0390 / 0.0387 / 0.0420
quant      mean 0.0445 +/- 0.0001 (n=2) | ens 0.0437 | d(base) -0.0021 [WIN] | corr(fbar, f_true) 0.903
           per-bin 0.0659 / 0.0479 / 0.0376 / 0.0362 / 0.0344 / 0.0387
fracquant  mean 0.0451 +/- 0.0006 (n=2) | ens 0.0439 | d(base) -0.0015 [noise] | corr(fbar, f_true) 0.968
           per-bin 0.0698 / 0.0481 / 0.0366 / 0.0374 / 0.0352 / 0.0370
frac: E>17 bins beating base by >0.002: none
quant: E>17 bins beating base by >0.002: [3, 4]
fracquant: E>17 bins beating base by >0.002: [4, 5]
